# Create dataset using preprocessed data

In [1]:
import csv

with open("processed-actual-outages.csv") as file:
    csv_reader = csv.DictReader(file)
    actual_data = list(csv_reader)

In [2]:
from datetime import datetime

from tqdm import tqdm

# convert dates to datetime objects, and PTID and Voltage to integers
date_keys = ["OutDatetime", "MinTimeStamp", "MaxTimeStamp"]

for row in tqdm(actual_data):
    row["PTID"] = int(row["PTID"])
    row["Voltage"] = int(row["Voltage"])
    for key in date_keys:
        row[key] = datetime.strptime(row[key], "%Y-%m-%d %H:%M:%S")

actual_data[0]

100%|██████████| 74063/74063 [00:00<00:00, 114946.69it/s]


{'PTID': 26053,
 'Name': 'MOUNTAIN-SWANROAD_115_104-3',
 'OutDatetime': datetime.datetime(2005, 1, 1, 0, 0),
 'MinTimeStamp': datetime.datetime(2012, 9, 11, 11, 52, 17),
 'MaxTimeStamp': datetime.datetime(2013, 8, 8, 8, 37, 17),
 'Voltage': 115,
 'FirstBus': 'MOUNTAIN',
 'SecondBus': 'SWANROAD',
 'OutageType': 'Planned'}

In [3]:
from pathlib import Path

output_path = Path("output")
output_path.mkdir(exist_ok=True)

# HYPER PARAMETERS

In [4]:
EVENT_WINDOW_HOURS = 6
MIN_YEAR = 2008

VOLTAGES = [69, 115, 132, 220, 345, 500, 735]
VOLTAGE_GROUP = {
    # 69 or less
    27: 69,
    34: 69,
    69: 69,
    # 155/120
    115: 115,
    120: 115,
    # 132/138
    132: 132,
    138: 132,
    # 220/230
    220: 220,
    230: 220,
    # 345
    345: 345,
    #500
    500: 500,
    # 735/765
    735: 735,
    765: 735,
}

INTERVALS_MINUTES = [15, 30, 60, 120, 240, 480]


In [5]:
actual_data = sorted([row for row in actual_data if row["MinTimeStamp"].year >= MIN_YEAR and row["OutDatetime"].year >= MIN_YEAR], key=lambda x: x["MinTimeStamp"])

In [6]:
# load graph
import igraph

g = igraph.Graph.Read_Pickle("res/outage_graph.pkl")


In [7]:
# load buses/nodes information
import json

bus_name_to_index_path = Path("res/bus_name_to_index.json")
bus_name_to_index = json.loads(bus_name_to_index_path.read_text())

In [8]:
from datetime import timedelta

import numpy as np


def create_sample(ref_row, window):
    ############
    # features #
    ############
    # simple counting features
    num_events = len(window)
    num_unique_ptids = len(set(row["PTID"] for row in window))
    voltage_group_list = {i: 0 for i in VOLTAGES}
    for row in window:
        voltage_group_list[VOLTAGE_GROUP[row["Voltage"]]] += 1
    voltage_group_list = [voltage_group_list[v] for v in VOLTAGES]
    num_planned = len([row for row in window if row["OutageType"] == "Planned"])
    num_auto = len([row for row in window if row["OutageType"] == "Auto"])
    bus_names = [row[bus] for row in window for bus in ["FirstBus", "SecondBus"]]
    num_unique_buses = len(set(bus_names))

    # fine-grained interval features
    fine_interval_features = []
    for dt in INTERVALS_MINUTES:
        dt = timedelta(minutes=dt)
        num_events_interval = len([row for row in window if ref_row["MinTimeStamp"] - row["MinTimeStamp"] < dt])
        fine_interval_features.append(num_events_interval)

    # graph based features
    node_degrees = np.array([g.degree(bus_name_to_index[bus_name]) for bus_name in bus_names])
    node_degrees_mean = node_degrees.mean().item()
    node_degrees_std = node_degrees.std().item()
    node_degrees_min = node_degrees.min().item()
    node_degrees_max = node_degrees.max().item()
    node_degrees_stats = [node_degrees_mean, node_degrees_std, node_degrees_min, node_degrees_max]

    # aggregate features
    features = [num_events, num_unique_ptids, *voltage_group_list, num_planned, num_auto, num_unique_buses, *fine_interval_features, *node_degrees_stats]

    ##########
    # labels #
    ##########
    # Auto/Planned labels
    is_auto = True if ref_row["OutageType"] == "Auto" else False

    # time to reference event
    time_to_event = ref_row["MinTimeStamp"] - max(row["MinTimeStamp"] for row in window)

    # aggregate labels
    labels = [is_auto, time_to_event.total_seconds()]

    return features, labels


In [9]:
from tqdm import tqdm

MAX_EVENTS_PER_WINDOW = 100

dataset = []

delta_time = timedelta(hours=EVENT_WINDOW_HOURS)
for i, ref_row in enumerate(tqdm(actual_data)):
    # exclude old data
    if ref_row["MinTimeStamp"].year < MIN_YEAR or ref_row["OutDatetime"].year < MIN_YEAR:
        continue

    # exclude planned event
    if ref_row["OutageType"] == "Planned":
        continue

    event_window_start = row["MinTimeStamp"] - delta_time
    window = [row for row in actual_data[max(i-MAX_EVENTS_PER_WINDOW, 0):i] if ref_row["MinTimeStamp"] - row["MinTimeStamp"] < delta_time]
    if window:
        dataset.append(create_sample(ref_row, window))

len(dataset)

100%|██████████| 70079/70079 [00:00<00:00, 284715.26it/s]


7314

In [10]:
features, labels = dataset[0]
features, labels

([1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 2, 0, 0, 0, 1, 1, 1, 3.0, 0.0, 3, 3],
 [True, 3600.0])

In [11]:
from sklearn.model_selection import train_test_split, KFold


def assign_fold_ids(dataset, test_size=0.20, n_splits=5, seed=42):
    """
    Assign fold_id to each sample in a dataset.

    fold_id = -1  -> held-out test set
    fold_id = 0-4 -> 5-fold CV folds inside the remaining train/validation set

    Parameters
    ----------
    dataset : list-like
        Dataset where each item is like:
            features, label = dataset[i]

    test_size : float
        Proportion of data reserved as held-out test set.

    n_splits : int
        Number of CV folds.

    seed : int
        Random seed for reproducibility.

    Returns
    -------
    fold_ids : np.ndarray
        Array of shape (len(dataset),), containing fold IDs.
    """

    n_samples = len(dataset)
    indices = np.arange(n_samples)

    # Initialise all samples as unassigned
    fold_ids = np.empty(n_samples, dtype=int)

    # Step 1: create held-out test set
    trainval_idx, test_idx = train_test_split(
        indices,
        test_size=test_size,
        random_state=seed,
        shuffle=True,
    )

    # Assign held-out test samples
    fold_ids[test_idx] = -1

    # Step 2: assign 5-fold IDs inside train/validation set
    kfold = KFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=seed,
    )

    for fold_id, (_, val_idx_relative) in enumerate(kfold.split(trainval_idx)):
        val_idx_absolute = trainval_idx[val_idx_relative]
        fold_ids[val_idx_absolute] = fold_id

    return fold_ids

In [12]:
from collections import Counter

feature_columns = [
    "num_events",
    "num_unique_ptids",
    *[f"num_{voltage}kv_lines" for voltage in VOLTAGES],
    "num_planned_outages",
    "num_auto_outages",
    "num_unique_buses",
    *[f"num_events_last_{minutes}_min" for minutes in INTERVALS_MINUTES],
    "node_degree_mean",
    "node_degree_std",
    "node_degree_min",
    "node_degree_max",
]
label_columns = ["label_is_auto", "label_time_to_event_seconds"]
columns = ["fold_id", *feature_columns, *label_columns]

fold_ids = assign_fold_ids(dataset)
dataset_csv_path = output_path / f"dataset_ws{EVENT_WINDOW_HOURS}.csv"

with dataset_csv_path.open("w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(columns)
    for fold_id, (features, labels) in zip(fold_ids, dataset):
        if len(features) != len(feature_columns) or len(labels) != len(label_columns):
            raise ValueError("Feature or label column count does not match the dataset sample shape.")
        writer.writerow([fold_id, *features, *labels])

dataset_csv_path, Counter(fold_ids)

(PosixPath('output/dataset_ws6.csv'),
 Counter({np.int64(-1): 1463,
          np.int64(0): 1171,
          np.int64(1): 1170,
          np.int64(3): 1170,
          np.int64(4): 1170,
          np.int64(2): 1170}))